## Seccion 1 - Configuracion e importaciones
En esta seccion realizamos los imports necesarios y declaramos valores constantes referentes a la ruta de nuestra base de datos y el url a scrappear

In [11]:
from bs4 import BeautifulSoup
import sqlite3
import requests
import time
import json
import re

BASE_URL = "https://books.toscrape.com/"
DB_PATH = "books.db"

## Seccion 2 - Diseño de modelo y creacion de tablas
![Diseño UML](./modelo-db-UML.svg)
Para crear una relacion N:M (muchos a muchos), es necesario crear una tabla intermedia, tener en cuenta que al utilizar 2 PK, realmente lo unico e irrepetible es la combinacion de (book_id, author_id), no cada uno de forma independiente.

### Tablas
- categories
- authors
- books
- book_author

In [12]:
# sqlite no tiene por defecto foreign keys, es necesario activar
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()
cursor.execute("PRAGMA foreign_keys = ON")
print("Conectado a db books.db")

# categorias
cursor.execute("""
    CREATE TABLE IF NOT EXISTS categories (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        name    TEXT UNIQUE NOT NULL
    )
""")

# authors
cursor.execute("""
    CREATE TABLE IF NOT EXISTS authors (
        id                  INTEGER PRIMARY KEY AUTOINCREMENT,
        name                TEXT NOT NULL,
        birth_year          INTEGER,
        country             TEXT,
        external_api_id     TEXT UNIQUE,
        total_known_works   INTEGER,
        api_source          TEXT,
        created_at          DATETIME DEFAULT CURRENT_TIMESTAMP
    )
""")

# books
cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        title           TEXT NOT NULL,
        price           REAL NOT NULL CHECK(price >= 0),
        rating          INTEGER NOT NULL CHECK(rating >= 0 AND rating <= 5),
        category_id     INTEGER NOT NULL,

        FOREIGN KEY (category_id) REFERENCES categories(id)
    )
""")

# book_author - tabla intermedia N:M
cursor.execute("""
    CREATE TABLE IF NOT EXISTS book_author (
        book_id     INTEGER NOT NULL,
        author_id   INTEGER NOT NULL,

        PRIMARY KEY (book_id, author_id),
        FOREIGN KEY (book_id) REFERENCES books(id),
        FOREIGN KEY (author_id) REFERENCES authors(id)
    )
""")

# crear indices minimos FK
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_category_id ON books(category_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_book_author_book_id ON book_author(book_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_book_author_author_id ON book_author(author_id)")


connection.commit()
connection.close()
print("Base de datos creada con exito")

Conectado a db books.db
Base de datos creada con exito


## Seccion 2 - Scraping de categorias
Scrapeamos la barra lateral donde se encuentran todas las categorias, asignamos un id autoincremente y cargamos en la tabla categories

In [13]:
# intentamos conectar con servidor
try :
    page = requests.get(BASE_URL, timeout=10)
    # elevamos excepcion si conexion no es exitosa
    page.raise_for_status()
    print("Conectado correctamente")

except requests.exceptions.HTTPError as err:
    print(f"Error HTTP: {err}")
except Exception as err:
    print(f"Otro error: {err}")

Conectado correctamente


In [14]:
# BeautifulSoup recibe html o xml en formato string o file-like object (archivo)
# y es necesario declarar parseador, por defecto html
soup = BeautifulSoup(page.text, "html.parser")
category_list = soup.find("ul", class_="nav nav-list").find("li").find_all("li")

# procesamos y agregamos a variable category_data, el nombre y el link
category_data = []
for category in category_list:
    name = category.text.strip()
    link = BASE_URL + category.a["href"]
    category_data.append((name, link))

# Conectamos con la DB
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

# OR IGNORE, para evitar cargar si ya esta cargado, por jupyter
for name, link in category_data:
    cursor.execute(
        "INSERT OR IGNORE INTO categories (name) VALUES (?)",
        (name,)
    )
    



cursor.execute("SELECT id, name FROM categories")
# crear diccionario con los key nombre y id value, para uso posterior 
category = {}
for row in cursor.fetchall():
    category[row[1]] = row[0]
    
connection.commit()
connection.close() 

print("Cantidad de categorias guardadas:", len(category))

Cantidad de categorias guardadas: 50


## Seccion 3 - Scraping de libros
Si bien podemos adquirir de que categoria es cada libro inspeccionando la pagina de destino con los detalles del libro, es mas sencillo realizar la carga de libros por categorias, donde directamente ya contamos con esa variable

In [15]:
# usamos category : dict para poder insertar con category_id
# tambien usamos category_data, ya que guardamos ahi las url
# creamos diccionario de conversion de ratings
ratings = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
books = []

for name, link in category_data:
    category_id = category[name]
    # creamos variable sin index, para poder concatenar futuras paginas
    link_no_index = link.replace("index.html", "")
    url_actual = link

    while url_actual:
        new_page = requests.get(url_actual, timeout=10)
        new_page.raise_for_status()
        soup_category = BeautifulSoup(new_page.text, "html.parser")
        articles = soup_category.find_all("article", class_="product_pod")

        for article in articles:
            title = article.h3.a["title"].strip()

            # con regex limpiamos 
            price_text = article.find("p", class_="price_color").text
            price = float(re.sub(r"[^0-9.]", "", price_text))

            # en la clase esta definido rating como segundo argumento
            rating_class = article.find("p", class_="star-rating")["class"][1]
            rating = ratings.get(rating_class, 0)

            books.append((title, price, rating, category_id))

        next_button = soup_category.find("li", class_="next")
        if next_button:
            href = next_button.a["href"]
            url_actual = link_no_index + href
        else:
            url_actual = None

print(f"{len(books)} libros extraidos")


1000 libros extraidos


In [16]:
# carga en db
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()

cursor.executemany("""
    INSERT INTO books (title, price, rating, category_id)
    VALUES (?, ?, ?, ?)
""", books)

connection.commit()

cursor.execute("SELECT COUNT(*) FROM books")
total = cursor.fetchone()[0]

connection.close() 


print(f"{total} libros guardados en la base de datos")

1000 libros guardados en la base de datos


## Enriquecimiento con openlibrary

In [17]:
# diccionario de conversion de gentilicio -> pais, usado para inferir nacionalidad
# a partir del campo bio que devuelve open library
mapeo_paises = {
    "american": "USA", "british": "UK", "english": "UK", "scottish": "UK",
    "welsh": "UK", "irish": "Ireland", "french": "France", "spanish": "Spain",
    "german": "Germany", "italian": "Italy", "canadian": "Canada",
    "australian": "Australia", "japanese": "Japan", "russian": "Russia",
    "chinese": "China", "mexican": "Mexico", "argentine": "Argentina",
    "colombian": "Colombia", "chilean": "Chile", "brazilian": "Brazil",
    "indian": "India", "swedish": "Sweden", "norwegian": "Norway",
    "dutch": "Netherlands", "polish": "Poland", "indonesian": "Indonesia"
}

# cache por author_key: si el mismo autor aparece en varios libros, evitamos repetir
# las llamadas de detalle (birth_date, bio) y obras (works) que son las mas lentas
cache_autores = {}

# Session reutiliza la conexion TCP entre llamadas, mas rapido que requests.get suelto
headers = {"User-Agent": "PenguinAcademyBot/1.0 (estudio_academico@ejemplo.com)"}
session = requests.Session()
session.headers.update(headers)


def limpiar_titulo(titulo):
    titulo = re.sub(r"\(.*?\)", "", titulo)
    titulo = re.sub(r"\s+", " ", titulo)
    return titulo.strip(" -:")


def extraer_pais_de_bio(autor_json):
    """
    Busca gentilicios dentro del campo bio de open library para inferir el pais.
    El campo bio puede venir como string o como dict {"value": "..."}.
    """
    bio_raw = autor_json.get("bio", "")
    if isinstance(bio_raw, dict):
        bio_texto = bio_raw.get("value", "")
    else:
        bio_texto = bio_raw or ""

    bio_texto = bio_texto.lower()

    for gentilicio, pais in mapeo_paises.items():
        if re.search(rf"\b{gentilicio}\b", bio_texto):
            return pais

    return None


def buscar_autor_openlibrary(titulo_original):
    """
    Busca el autor de un libro en Open Library a partir del titulo.
    Devuelve un diccionario con los datos enriquecidos, o valores NULL si no se encuentra.
    """
    titulo_limpio = limpiar_titulo(titulo_original)

    datos = {
        "name": None,
        "birth_year": None,
        "country": None,
        "external_api_id": None,
        "total_known_works": None,
        "api_source": None
    }

    try:
        params = {"title": titulo_limpio, "limit": 1, "fields": "title,author_name,author_key"}
        response = session.get("https://openlibrary.org/search.json", params=params, timeout=8)
        response.raise_for_status()
        resultado = response.json()

        docs = resultado.get("docs", [])
        if not docs or not docs[0].get("author_name"):
            datos["api_source"] = "not_found"
            return datos

        doc = docs[0]
        author_key = doc.get("author_key", [None])[0]
        author_name = doc.get("author_name", [None])[0]

        if not author_key or not author_name:
            datos["api_source"] = "not_found"
            return datos

        # si ya procesamos este autor antes (por otro libro), reusamos directo, sin nuevas llamadas
        if author_key in cache_autores:
            return cache_autores[author_key]

        datos["name"] = author_name
        datos["external_api_id"] = author_key
        datos["api_source"] = "open_library"

        clean_key = author_key.replace("/authors/", "")

        # detalle del autor (fecha de nacimiento + pais desde bio)
        try:
            author_response = session.get(f"https://openlibrary.org/authors/{clean_key}.json", timeout=8)
            if author_response.ok:
                autor_json = author_response.json()

                birth_raw = autor_json.get("birth_date", "")
                match = re.search(r"\b(1[0-9]{3}|20[0-2][0-9])\b", birth_raw)
                datos["birth_year"] = int(match.group(1)) if match else None

                datos["country"] = extraer_pais_de_bio(autor_json)
        except requests.exceptions.RequestException:
            pass

        # cantidad de obras conocidas
        try:
            search_response = session.get(
                "https://openlibrary.org/search/authors.json",
                params={"q": f"key:(/authors/{clean_key})"}, timeout=8
            )
            if search_response.ok:
                docs = search_response.json().get("docs", [])
                if docs:
                    datos["total_known_works"] = docs[0].get("work_count")
        except requests.exceptions.RequestException:
            pass

        # guardamos en cache por author_key, no por titulo
        cache_autores[author_key] = datos

    except requests.exceptions.Timeout:
        datos["api_source"] = "error_timeout"
    except requests.exceptions.HTTPError:
        datos["api_source"] = "error_http"
    except requests.exceptions.RequestException:
        datos["api_source"] = "error_conexion"

    return datos

In [18]:
connection = sqlite3.connect(DB_PATH, timeout=10)
cursor = connection.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

cursor.execute("SELECT id, title FROM books ORDER BY id")
libros_db = cursor.fetchall()
total_libros = len(libros_db)

tiempo_inicial = time.time()
encontrados = 0
no_encontrados = 0

for i, (book_id, title) in enumerate(libros_db, 1):
    info_autor = buscar_autor_openlibrary(title)

    if info_autor["name"]:
        cursor.execute("""
            INSERT INTO authors (name, birth_year, country, external_api_id, total_known_works, api_source)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(external_api_id) DO UPDATE SET
                birth_year = COALESCE(excluded.birth_year, authors.birth_year),
                total_known_works = COALESCE(excluded.total_known_works, authors.total_known_works)
        """, (info_autor["name"], info_autor["birth_year"], info_autor["country"],
              info_autor["external_api_id"], info_autor["total_known_works"], info_autor["api_source"]))

        cursor.execute("SELECT id FROM authors WHERE external_api_id = ?", (info_autor["external_api_id"],))
        author_row = cursor.fetchone()

        if author_row:
            cursor.execute(
                "INSERT OR IGNORE INTO book_author (book_id, author_id) VALUES (?, ?)",
                (book_id, author_row[0])
            )
        encontrados += 1
    else:
        no_encontrados += 1

    if i % 50 == 0 or i == total_libros:
        elapsed = round(time.time() - tiempo_inicial, 1)
        print(f"  {i}/{total_libros} libros procesados... ({elapsed}s transcurridos)")

connection.commit()
connection.close()

tiempo_final = time.time()
print(f"\nProceso completado en {round(tiempo_final - tiempo_inicial, 2)} segundos")
print(f"Autores encontrados: {encontrados}/{total_libros} ({round(encontrados/total_libros*100, 1)}%)")
print(f"Autores no encontrados: {no_encontrados}/{total_libros} ({round(no_encontrados/total_libros*100, 1)}%)")

  50/1000 libros procesados... (47.5s transcurridos)
  100/1000 libros procesados... (87.6s transcurridos)
  150/1000 libros procesados... (116.1s transcurridos)
  200/1000 libros procesados... (163.2s transcurridos)
  250/1000 libros procesados... (208.5s transcurridos)
  300/1000 libros procesados... (253.5s transcurridos)
  350/1000 libros procesados... (288.5s transcurridos)
  400/1000 libros procesados... (320.7s transcurridos)
  450/1000 libros procesados... (356.9s transcurridos)
  500/1000 libros procesados... (396.9s transcurridos)
  550/1000 libros procesados... (438.1s transcurridos)
  600/1000 libros procesados... (478.7s transcurridos)
  650/1000 libros procesados... (518.1s transcurridos)
  700/1000 libros procesados... (558.0s transcurridos)
  750/1000 libros procesados... (598.3s transcurridos)
  800/1000 libros procesados... (640.1s transcurridos)
  850/1000 libros procesados... (679.6s transcurridos)
  900/1000 libros procesados... (708.2s transcurridos)
  950/1000 li

## SQL Querys

In [ ]:
connection = sqlite3.connect(DB_PATH, timeout=10)
cursor = connection.cursor()

# Libros con mas de 3 estrellas por menos de 10 euros
cursor.execute("""
    SELECT title, price, rating
    FROM books
    WHERE rating > 3 AND price < 10
    ORDER BY rating DESC, price ASC
""")
print("Libros con mas de 3 estrellas por menos de 10 euros")
for row in cursor.fetchall():
    print(row)

# Autor con peor promedio de rating 
cursor.execute("""
    SELECT a.name, AVG(b.rating) AS promedio_rating, COUNT(b.id) AS cantidad_libros
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    JOIN books b ON ba.book_id = b.id
    GROUP BY a.id
    HAVING COUNT(b.id) >= 5
    ORDER BY promedio_rating ASC
    LIMIT 1
""")
print("Autor con peor promedio de rating")
print(cursor.fetchone())

# Categoria con mayor precio promedio
cursor.execute("""
    SELECT c.name, AVG(b.price) AS precio_promedio
    FROM categories c
    JOIN books b ON c.id = b.category_id
    GROUP BY c.id
    ORDER BY precio_promedio DESC
    LIMIT 1
""")
print("Categoria con mayor precio promedio")
print(cursor.fetchone())

# top 5 autores con mas libros
cursor.execute("""
    SELECT a.name, COUNT(ba.book_id) AS cantidad_libros
    FROM authors a
    JOIN book_author ba ON a.id = ba.author_id
    GROUP BY a.id
    ORDER BY cantidad_libros DESC
    LIMIT 5
""")
print ("top 5 autores con mas libros")
for row in cursor.fetchall():
    print(row)

# pais con mas libros con rating mayor a 3
cursor.execute("""
    SELECT a.country, COUNT(b.id) AS cantidad_libros
    FROM books b
    JOIN book_author ba ON b.id = ba.book_id
    JOIN authors a ON ba.author_id = a.id
    WHERE b.rating > 3 AND a.country IS NOT NULL
    GROUP BY a.country
    ORDER BY cantidad_libros DESC
""")
print("pais con mas libros con rating mayor a 3")
for row in cursor.fetchall():
    print(row)


connection.close()

Libros con mas de 3 estrellas por menos de 10 euros
Autor con peor promedio de rating
('Sophie Kinsella', 2.0, 5)
Categoria con mayor precio promedio
('Suspense', 58.33)
top 5 autores con mas libros
('Stephen King', 15)
('Worth Books', 10)
('J.K. Rowling', 8)
('Cassandra Clare', 7)
('Sophie Kinsella', 5)
pais con mas libros con rating mayor a 3
('USA', 61)
('UK', 28)
('Canada', 4)
('France', 2)
('Netherlands', 1)
('Italy', 1)
('Ireland', 1)
('Germany', 1)


 ## Indexación y Performance

In [10]:
connection = sqlite3.connect(DB_PATH, timeout=10)
cursor = connection.cursor()

cursor.execute("DROP INDEX IF EXISTS idx_books_price")
connection.commit()
print("Indice eliminado, listo para repetir la demostracion")

# usamos price, que no tiene indice propio (solo PK e implicitos de FK/UNIQUE)
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM books WHERE price > 30 ORDER BY price")
print("Plan SIN indice:")
for row in cursor.fetchall():
    print(row)

tiempo_inicial = time.time()
for _ in range(200):  # repetimos varias veces para que el tiempo sea medible
    cursor.execute("SELECT * FROM books WHERE price > 30 ORDER BY price")
    cursor.fetchall()
tiempo_sin_indice = time.time() - tiempo_inicial
print(f"Tiempo SIN indice (200 ejecuciones): {round(tiempo_sin_indice, 4)} segundos")

# creamos indice
cursor.execute("CREATE INDEX IF NOT EXISTS idx_books_price ON books(price)")
connection.commit()
print("Indice creado: idx_books_price")

# misma consulta pero con indice
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM books WHERE price > 30 ORDER BY price")
print("Plan CON indice:")
for row in cursor.fetchall():
    print(row)

tiempo_inicial = time.time()
for _ in range(200):
    cursor.execute("SELECT * FROM books WHERE price > 30 ORDER BY price")
    cursor.fetchall()
tiempo_con_indice = time.time() - tiempo_inicial
print(f"Tiempo CON indice (200 ejecuciones): {round(tiempo_con_indice, 4)} segundos")

connection.close()

# tabla comparativa de rendimiento
print(f"{'Escenario':<20}{'Tiempo (s)':<15}")
print(f"{'Sin indice':<20}{round(tiempo_sin_indice, 4):<15}")
print(f"{'Con indice':<20}{round(tiempo_con_indice, 4):<15}")
mejora = round((1 - tiempo_con_indice / tiempo_sin_indice) * 100, 1)
print(f"\nMejora: {mejora}%")

Indice eliminado, listo para repetir la demostracion
Plan SIN indice:
(3, 0, 216, 'SCAN books')
(17, 0, 0, 'USE TEMP B-TREE FOR ORDER BY')
Tiempo SIN indice (200 ejecuciones): 0.0778 segundos
Indice creado: idx_books_price
Plan CON indice:
(4, 0, 203, 'SEARCH books USING INDEX idx_books_price (price>?)')
Tiempo CON indice (200 ejecuciones): 0.067 segundos
Escenario           Tiempo (s)     
Sin indice          0.0778         
Con indice          0.067          

Mejora: 13.9%
